# Lecture 5.3: PostToolUse, Changing What Claude Sees

A PostToolUse hook runs **after** a tool has already run, so it cannot undo the tool. What it can do is control what Claude is told about the result. In this notebook you will:

1. Replace a tool's output with `updatedToolOutput`.
2. Add a note with `additionalContext`.

The one rule to remember: for built-in tools, keep the tool's own output shape, or the change is silently ignored.

Live agent output varies from run to run. Look at the checks each cell prints rather than expecting identical wording.

## Cell 1: Install the Claude Agent SDK

This notebook uses one package, `claude-agent-sdk`. It is the Python library that lets you run Claude as an agent and attach hooks to it. The wheel also bundles the Claude Code CLI that does the actual agent work, so there is nothing else to install.

The version is **pinned** so that every example behaves exactly as you see it in the video. If the package is already present in your Colab session, pip checks it, finds the pinned version and moves on quickly. The `-q` flag keeps the output quiet.

You can install the latest version or any version you prefer by editing the last line. Newer versions may behave differently from the video.

In [1]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.163 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.3 MB/s eta 0:00:00


## Cell 2: Set up your API key

The agent needs an Anthropic API key to call Claude. In Colab, the safe place for it is **Secrets**, so the key never appears in the notebook itself.

**If you do not have a key yet:** create one in the Claude Console at https://platform.claude.com/ under **API Keys**.

**Add the key as a Colab secret:**

1. Click the **key icon** in the left sidebar.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value.
5. Switch **Notebook access** on.

The cell below reads the secret with `userdata.get` and stores it in the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it.

**Running locally instead of Colab?** Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before you start Python.

In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

`MODEL_NAME` holds the model every agent in this notebook will use. We pass it as `model=MODEL_NAME` in each `ClaudeAgentOptions`, so **changing this one line changes the model for the entire notebook**.

We use `claude-haiku-4-5`, the fast and economical default for this course. Model names change as new releases arrive, so the comment points to the official models page where you can find the latest.

In [3]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Cell 4: Create the sample files

Both demos need a small workspace under `/content`, and we create it from code so you can see exactly what is in it.

| Path | Used by | Contents |
|---|---|---|
| `/content/redact_demo/customers.csv` | Demo A and Demo B | Four **invented** customers with an id, name, email and city |

The email addresses use the reserved `example.com` domain, so none of them belong to a real person. Demo A will intercept the output of a command that prints this file, so we need a file with emails in it.

In [4]:
import os

os.makedirs("/content/redact_demo", exist_ok=True)

customers_csv = '''id,name,email,city
1,Ana Rao,ana.rao@example.com,Lisbon
2,Ben Okafor,ben.okafor@example.com,Leeds
3,Chloe Tan,chloe.tan@example.com,Perth
4,Dev Patel,dev.patel@example.com,Austin
'''

with open("/content/redact_demo/customers.csv", "w") as f:
    f.write(customers_csv)

print("Sample files created.")

Sample files created.


## Cell 5: Verify the sample files

Before any agent runs, we confirm what is on disk. This cell prints the directory tree and the contents of the CSV. Later cells use the disk as **ground truth**: instead of trusting what an agent says it did, we look at the filesystem.

In [5]:
for folder, subfolders, files in os.walk("/content/redact_demo"):
    print(folder)
    for name in files:
        print("   ", name)

print()
print("--- customers.csv ---")
with open("/content/redact_demo/customers.csv") as f:
    print(f.read())

/content/redact_demo
    customers.csv

--- customers.csv ---
id,name,email,city
1,Ana Rao,ana.rao@example.com,Lisbon
2,Ben Okafor,ben.okafor@example.com,Leeds
3,Chloe Tan,chloe.tan@example.com,Perth
4,Dev Patel,dev.patel@example.com,Austin



## Cell 6: Demo A, step 1: inspect the real `tool_response`

A **PostToolUse hook** runs *after* a tool has finished. It receives the tool's result in `input_data["tool_response"]`. Before we change anything, we need to know what that object really looks like, so this first hook only **prints** it and returns `{}`, which means "change nothing".

| Piece | What it does |
|---|---|
| `async def inspect_hook(input_data, tool_use_id, context)` | The hook callback. The `context` argument is reserved and is always `None` in Python. |
| `input_data["tool_response"]` | The result of the tool call, as the hook sees it. |
| `return {}` | Leaves the output untouched. |
| `HookMatcher(matcher="Bash", hooks=[inspect_hook])` | Runs the hook only for the `Bash` tool. Matchers match **tool names**, never file paths. |
| `ClaudeAgentOptions(hooks={"PostToolUse": [...]})` | Registers hooks for the event. Hooks work with `query()`. |
| `allowed_tools=["Bash"]` | Least privilege: this demo needs only Bash. |

**What to look for.** The `HOOK saw` lines show the real shape. The SDK source mentions `stdout`, `stderr` and `interrupted` for Bash. In our test run (SDK 0.2.163) the dictionary also had `isImage` and `noOutputExpected`, so it had five keys. Your version may differ, so read what your run actually prints.

We also print every `ToolResultBlock` we find in the stream. Remember the mental model: `query()` returns an **async stream of messages**, and a tool result arrives inside a `UserMessage`, not an `AssistantMessage`. The final answer arrives on the `ResultMessage`.

In [6]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    HookMatcher,
    ResultMessage,
    UserMessage,
    ToolResultBlock,
)


async def inspect_hook(input_data, tool_use_id, context):
    response = input_data["tool_response"]
    print("HOOK saw type:", type(response).__name__)
    if isinstance(response, dict):
        print("HOOK saw keys:", list(response.keys()))
    print("HOOK saw repr:", repr(response)[:300])
    return {}


options = ClaudeAgentOptions(
    allowed_tools=["Bash"],
    model=MODEL_NAME,
    cwd="/content",
    hooks={
        "PostToolUse": [
            HookMatcher(matcher="Bash", hooks=[inspect_hook]),
        ]
    },
)

prompt = (
    "Use the Bash tool to print the file /content/redact_demo/customers.csv. "
    "Then tell me how many customers are listed."
)

async for message in query(prompt=prompt, options=options):
    if isinstance(message, UserMessage) and isinstance(message.content, list):
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                print("CLAUDE SAW (ToolResultBlock):", repr(block.content)[:300])
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print("FINAL ANSWER:", message.result)

HOOK saw type: dict
HOOK saw keys: ['stdout', 'stderr', 'interrupted', 'isImage', 'noOutputExpected']
HOOK saw repr: {'stdout': 'id,name,email,city\n1,Ana Rao,ana.rao@example.com,Lisbon\n2,Ben Okafor,ben.okafor@example.com,Leeds\n3,Chloe Tan,chloe.tan@example.com,Perth\n4,Dev Patel,dev.patel@example.com,Austin', 'stderr': '', 'interrupted': False, 'isImage': False, 'noOutputExpected': False}
CLAUDE SAW (ToolResultBlock): 'id,name,email,city\n1,Ana Rao,ana.rao@example.com,Lisbon\n2,Ben Okafor,ben.okafor@example.com,Leeds\n3,Chloe Tan,chloe.tan@example.com,Perth\n4,Dev Patel,dev.patel@example.com,Austin'
FINAL ANSWER: The file contains **4 customers**:

1. Ana Rao (ana.rao@example.com, Lisbon)
2. Ben Okafor (ben.okafor@example.com, Leeds)
3. Chloe Tan (chloe.tan@example.com, Perth)
4. Dev Patel (dev.patel@example.com, Austin)


## Cell 7: Demo A, step 2: redact emails with `updatedToolOutput`

Now we change what Claude sees. The hook returns `updatedToolOutput` inside `hookSpecificOutput`, and the SDK uses that value **in place of** the original output.

**The one rule.** For built-in tools like Bash, the replacement must keep the tool's own output shape. So we do not build a new object from scratch. We **copy** the real `tool_response` dictionary and change only the `stdout` value. Everything else, such as `stderr`, `interrupted` and any other fields your version includes, stays exactly as it was.

| Piece | What it does |
|---|---|
| `mask_emails(text)` | Turns `ana.rao@example.com` into `a***@example.com` by keeping the first character of the local part. |
| `dict(response)` | A copy of the original dictionary, so the schema is preserved. |
| `updated["stdout"] = ...` | Changes only the field we mean to change. |
| `hookEventName: "PostToolUse"` | **Required** inside `hookSpecificOutput`. It tells the SDK which event this output belongs to. |
| `seen` | A dictionary where the hook records what it saw and what it returned, so we can compare them afterwards. |

The hook returns `{}` if the response does not look like a dictionary with a `stdout` field. That guard keeps the demo safe if your step 1 output showed a different shape.

**What to look for.** The goal prompt asks Claude to quote the first customer's email *exactly as the tool printed it*. If the replacement reached Claude, the answer should contain the masked form. The final check compares the answer against the original address. As always with live agents, the exact wording of the answer varies from run to run, so look at the checks rather than the phrasing.

In [7]:
import re

ORIGINAL_EMAIL = "ana.rao@example.com"
seen = {}


def mask_emails(text):
    return re.sub(r"\b([A-Za-z0-9])[A-Za-z0-9._%+-]*@", r"\1***@", text)


async def redact_hook(input_data, tool_use_id, context):
    response = input_data["tool_response"]
    if not isinstance(response, dict) or "stdout" not in response:
        return {}
    updated = dict(response)
    updated["stdout"] = mask_emails(response["stdout"])
    seen["before"] = response["stdout"]
    seen["after"] = updated["stdout"]
    return {
        "hookSpecificOutput": {
            "hookEventName": "PostToolUse",
            "updatedToolOutput": updated,
        }
    }


options = ClaudeAgentOptions(
    allowed_tools=["Bash"],
    model=MODEL_NAME,
    cwd="/content",
    hooks={
        "PostToolUse": [
            HookMatcher(matcher="Bash", hooks=[redact_hook]),
        ]
    },
)

prompt = (
    "Use the Bash tool to print the file /content/redact_demo/customers.csv. "
    "Then quote the first customer's email EXACTLY as the tool printed it."
)

final_answer = ""
async for message in query(prompt=prompt, options=options):
    if isinstance(message, UserMessage) and isinstance(message.content, list):
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                print("CLAUDE SAW (ToolResultBlock):", repr(block.content)[:300])
    if isinstance(message, ResultMessage) and message.subtype == "success":
        final_answer = message.result

print()
print("HOOK saw    :", repr(seen.get("before"))[:120])
print("HOOK returned:", repr(seen.get("after"))[:120])
print("FINAL ANSWER:", final_answer)
print()
print("Original address in answer? ", ORIGINAL_EMAIL in final_answer)
print("Masked address in answer?   ", "a***@example.com" in final_answer)

CLAUDE SAW (ToolResultBlock): 'id,name,email,city\n1,Ana Rao,a***@example.com,Lisbon\n2,Ben Okafor,b***@example.com,Leeds\n3,Chloe Tan,c***@example.com,Perth\n4,Dev Patel,d***@example.com,Austin'

HOOK saw    : 'id,name,email,city\n1,Ana Rao,ana.rao@example.com,Lisbon\n2,Ben Okafor,ben.okafor@example.com,Leeds\n3,Chloe Tan,chloe.
HOOK returned: 'id,name,email,city\n1,Ana Rao,a***@example.com,Lisbon\n2,Ben Okafor,b***@example.com,Leeds\n3,Chloe Tan,c***@example.co
FINAL ANSWER: The first customer's email EXACTLY as printed is: `a***@example.com`

Original address in answer?  False
Masked address in answer?    True


## Cell 8: Demo A, step 3: the silent failure

This cell runs the same demo, but the hook returns the **wrong shape**. Instead of a dictionary with `stdout`, `stderr` and `interrupted`, it returns a plain string.

The SDK source says that for built-in tools, an `updatedToolOutput` that does not match the tool's output schema is not applied, and the **original output is kept**. It also says this happens silently. So we expect no crash and no error message, only an output that never changed.

| Piece | What it does |
|---|---|
| `bad_shape_hook` | Returns `updatedToolOutput` as a plain string, which is not the Bash shape. |
| `ORIGINAL_EMAIL in final_answer` | Our ground truth. If the original address is in the answer, the replacement did not reach Claude. |

**What to look for.** First confirm the `HOOK fired` line printed, so you know the hook ran. Then read the `CLAUDE SAW` line and the final check. This is the behaviour documented in the source, and your run is the confirmation. If your result differs, trust what you observe and note it.

**Why this matters.** A silent failure is easy to miss, because your hook ran and returned something and nothing complained. This is why step 1 prints the real shape first, and why step 2 copies it.

In [8]:
async def bad_shape_hook(input_data, tool_use_id, context):
    response = input_data["tool_response"]
    if not isinstance(response, dict) or "stdout" not in response:
        return {}
    print("HOOK fired: returning a plain string as updatedToolOutput")
    return {
        "hookSpecificOutput": {
            "hookEventName": "PostToolUse",
            "updatedToolOutput": mask_emails(response["stdout"]),
        }
    }


options = ClaudeAgentOptions(
    allowed_tools=["Bash"],
    model=MODEL_NAME,
    cwd="/content",
    hooks={
        "PostToolUse": [
            HookMatcher(matcher="Bash", hooks=[bad_shape_hook]),
        ]
    },
)

final_answer = ""
async for message in query(prompt=prompt, options=options):
    if isinstance(message, UserMessage) and isinstance(message.content, list):
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                print("CLAUDE SAW (ToolResultBlock):", repr(block.content)[:300])
    if isinstance(message, ResultMessage) and message.subtype == "success":
        final_answer = message.result

print()
print("FINAL ANSWER:", final_answer)
print()
print("Original address in answer? ", ORIGINAL_EMAIL in final_answer)
print("Masked address in answer?   ", "a***@example.com" in final_answer)

HOOK fired: returning a plain string as updatedToolOutput
CLAUDE SAW (ToolResultBlock): 'id,name,email,city\n1,Ana Rao,ana.rao@example.com,Lisbon\n2,Ben Okafor,ben.okafor@example.com,Leeds\n3,Chloe Tan,chloe.tan@example.com,Perth\n4,Dev Patel,dev.patel@example.com,Austin'

FINAL ANSWER: The first customer's email EXACTLY as the tool printed it is:

`ana.rao@example.com`

Original address in answer?  True
Masked address in answer?    False


## Cell 9: Demo B: add a note with `additionalContext`

`updatedToolOutput` replaces the result. `additionalContext` is gentler: it **adds** a short note next to the result. The hook returns it in the same `hookSpecificOutput` block, with `hookEventName` set to `"PostToolUse"`.

| Field | Effect |
|---|---|
| `updatedToolOutput` | Replaces the output before Claude sees it. |
| `additionalContext` | Appends a note for Claude alongside the output. |

Here the note tells Claude that the folder holds invented sample data. The prompt then asks Claude whether it received any note about the data.

**What to look for.** Read the final answer. Claude may or may not mention the note, and it may mention it on one run and not on the next. Treat `additionalContext` as a **hint Claude can use**, not as a guaranteed instruction. Run the cell more than once if you want to see the variation.

In [9]:
async def context_hook(input_data, tool_use_id, context):
    return {
        "hookSpecificOutput": {
            "hookEventName": "PostToolUse",
            "additionalContext": "Note: every record in this folder is invented sample data.",
        }
    }


options = ClaudeAgentOptions(
    allowed_tools=["Bash"],
    model=MODEL_NAME,
    cwd="/content",
    hooks={
        "PostToolUse": [
            HookMatcher(matcher="Bash", hooks=[context_hook]),
        ]
    },
)

prompt_b = (
    "Use the Bash tool to list the files in /content/redact_demo. "
    "Then tell me whether you were given any note about the data in that folder."
)

async for message in query(prompt=prompt_b, options=options):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print("FINAL ANSWER:", message.result)

FINAL ANSWER: The `/content/redact_demo` directory contains one file:
- `customers.csv` (179 bytes)

**Yes, I was given a note about the data**: The system reminder states that **"every record in this folder is invented sample data."** This means all the data in the customers.csv file is synthetic/generated test data, not real customer information.
